# Exploded call hierarchy

**Entry point:** the selected call whose consequences are expanded below.

- **Call site:** [stream_to_datasources.py](pyPhoTimeline/pypho_timeline/rendering/datasources/stream_to_datasources.py#L406)

In [ ]:
# Call site: pyPhoTimeline/.../stream_to_datasources.py (line 406)
obj: LabRecorderXDF = LabRecorderXDF.init_from_lab_recorder_xdf_file(a_xdf_file=xdf_file_path, should_load_full_file_data=True)

---

## Level 0 — Call site

- **File:** [stream_to_datasources.py](pyPhoTimeline/pypho_timeline/rendering/datasources/stream_to_datasources.py#L406)
- **Result:** `obj` is a `LabRecorderXDF` with `stream_infos`, `streams_timestamp_dfs`, `datasets`, `datasets_dict` populated when `should_load_full_file_data=True`.

---

## Level 1 — `LabRecorderXDF.init_from_lab_recorder_xdf_file`

- **Defined in:** [xdf_files.py](PhoPyMNEHelper/src/phopymnehelper/xdf_files.py#L781)
- **Calls (in order):**
  1. `init_basic_from_lab_recorder_xdf_file(...)` → returns `_obj` with `xdf_streams`, `xdf_header`, `file_datetime` set.
  2. If `should_load_full_file_data` is **False**: `_obj.perform_process_xdf_streams(...)` (metadata only).
  3. If **True**: `_obj.perform_load_xdf_streams(...)` (full load).
  4. Return `_obj`.

In [ ]:
# PhoPyMNEHelper/src/phopymnehelper/xdf_files.py (LabRecorderXDF, L781)
@classmethod
def init_from_lab_recorder_xdf_file(cls, a_xdf_file: Path, should_load_full_file_data: bool = True, debug_print: bool = False):
    _obj = cls.init_basic_from_lab_recorder_xdf_file(a_xdf_file=a_xdf_file, skipped_stream_names=skipped_stream_names, debug_print=debug_print)
    if not should_load_full_file_data:
        stream_infos, streams_timestamp_dfs = _obj.perform_process_xdf_streams(debug_print=debug_print)
        # ...
    else:
        stream_infos, streams_timestamp_dfs, datasets, datasets_dict = _obj.perform_load_xdf_streams(debug_print=debug_print)
    return _obj

---

## Level 2a — `init_basic_from_lab_recorder_xdf_file`

- **Defined in:** [xdf_files.py](PhoPyMNEHelper/src/phopymnehelper/xdf_files.py#L396)
- **Calls:** `pyxdf.load_xdf(...)` → `cls(...)` (constructs `LabRecorderXDF`) → `datetime.strptime(header['info']['datetime'][0], "%Y-%m-%dT%H:%M:%S%z")` → `.astimezone(timezone.utc)` → assign `_obj.file_datetime` → return `_obj`.

In [ ]:
# PhoPyMNEHelper/src/phopymnehelper/xdf_files.py (L396–405)
@classmethod
def init_basic_from_lab_recorder_xdf_file(cls, a_xdf_file: Path, skipped_stream_names: List[str] = None, debug_print=False, **kwargs):
    streams, header = pyxdf.load_xdf(a_xdf_file, synchronize_clocks=True, handle_clock_resets=True, dejitter_timestamps=False, verbose=True)
    _obj = cls(xdf_file_path=a_xdf_file, xdf_streams=streams, xdf_header=header, skipped_stream_names=skipped_stream_names, **kwargs)
    _obj.file_datetime = datetime.strptime(header['info']['datetime'][0], "%Y-%m-%dT%H:%M:%S%z").astimezone(timezone.utc)
    return _obj

---

## Level 2b — `perform_load_xdf_streams` (when full load)

- **Defined in:** [xdf_files.py](PhoPyMNEHelper/src/phopymnehelper/xdf_files.py#L565)
- **Per stream:**
  1. [parse_and_add_lsl_outlet_info_from_desc](PhoPyLSLhelper/src/phopylslhelper/easy_time_sync.py#L104) (adds `stream_start_datetime`, etc.)
  2. `mne.create_info(...)`
  3. **`info.set_meas_date(self.file_datetime)`** ([meas_info.py](mne-python/mne/_fiff/meas_info.py#L834)) → can trigger MNE `_check_dt` (Level 4)
  4. `mne.io.RawArray(data, info)` or `mne.Annotations(..., orig_time=stream_info_dict['stream_start_datetime'])` ([annotations.py](mne-python/mne/annotations.py#L393) → `_handle_meas_date` → `_check_dt`)
  5. Build `stream_infos` DataFrame, then return.

In [ ]:
# PhoPyMNEHelper/src/phopymnehelper/xdf_files.py (L631–632, L721–722, L607, L731)
stream_info_dict = EasyTimeSyncParsingMixin.parse_and_add_lsl_outlet_info_from_desc(
    desc_info_dict=desc_info_dict, stream_info_dict=stream_info_dict, should_fail_on_missing=False
)
info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=ch_types)
info = info.set_meas_date(self.file_datetime)  # ← can raise ValueError (Level 4)
raw = mne.io.RawArray(data, info)
# or for logger streams:
raw = mne.Annotations(onset=..., duration=..., description=..., orig_time=stream_info_dict['stream_start_datetime'])  # ← same check

---

## Level 3 — `perform_process_xdf_streams` (lightweight path)

- **Defined in:** [xdf_files.py](PhoPyMNEHelper/src/phopymnehelper/xdf_files.py#L409)
- Same per-stream metadata and [parse_and_add_lsl_outlet_info_from_desc](PhoPyLSLhelper/src/phopylslhelper/easy_time_sync.py#L104); **no** `mne.create_info` / `set_meas_date` / `RawArray` / `Annotations`.
- **Returns:** `(stream_infos, streams_timestamp_dfs)`.

---

## Level 4 — MNE datetime validation (source of the TODO error)

**Error:** `ValueError: Date must be datetime object in UTC: datetime.datetime(2026, 3, 1, 2, 9, 18, tzinfo=<UTC>)`

- **Raised in:** [_check_dt](mne-python/mne/utils/numerics.py#L1014) in [numerics.py](mne-python/mne/utils/numerics.py#L1014).
- **Call chain:** `info.set_meas_date(...)` → [Info.set_meas_date](mne-python/mne/_fiff/meas_info.py#L834) → [meas_date = _handle_meas_date(meas_date)](mne-python/mne/annotations.py#L1192) → **[ _check_dt(meas_date)](mne-python/mne/utils/numerics.py#L1022)** (via [_handle_meas_date](mne-python/mne/annotations.py#L1223)); or `Annotations(..., orig_time=...)` → [Annotations](mne-python/mne/annotations.py#L393) → `_handle_meas_date(orig_time)` → **`_check_dt`**.
- **Requirement:** `dt` must be a `datetime`, with `dt.tzinfo` such that **either** `dt.tzinfo is timezone.utc` (stdlib) **or** `dt.tzinfo.zone == pytz.timezone("UTC").zone`. Pandas `Timestamp` or other UTC types can fail this identity check.
- **Fix:** Normalize before passing to MNE: e.g. `datetime.fromtimestamp(dt.timestamp(), tz=timezone.utc)` or ensure stdlib `timezone.utc` is used (e.g. in PhoPyMNEHelper set `file_datetime` and any `stream_start_datetime` to a datetime with `tzinfo=datetime.timezone.utc`).

In [ ]:
# mne-python/mne/utils/numerics.py (L1014–1022)
def _check_dt(dt):
    if (
        not isinstance(dt, datetime)
        or dt.tzinfo is None
        or not ((dt.tzinfo is timezone.utc) or (dt.tzinfo.zone is pytz.timezone("UTC").zone))
    ):
        raise ValueError(f"Date must be datetime object in UTC: {repr(dt)}")

---

## Summary diagram

```
stream_to_datasources.py:406
  LabRecorderXDF.init_from_lab_recorder_xdf_file(...)
    ├── init_basic_from_lab_recorder_xdf_file(...)
    │     ├── pyxdf.load_xdf(...)
    │     ├── cls(...)
    │     ├── datetime.strptime(...)
    │     └── .astimezone(timezone.utc)
    └── perform_load_xdf_streams(...)   [when should_load_full_file_data=True]
          for each stream:
            ├── EasyTimeSyncParsingMixin.parse_and_add_lsl_outlet_info_from_desc(...)
            ├── mne.create_info(...)
            ├── info.set_meas_date(self.file_datetime)  ──► _handle_meas_date → _check_dt  [ValueError here]
            ├── mne.io.RawArray(data, info)
            └── or mne.Annotations(..., orig_time=...)  ──► _handle_meas_date → _check_dt
```

**Consequence:** Full load of the XDF into `LabRecorderXDF` with MNE `Raw`/`Annotations` per stream; datetimes passed to MNE must satisfy `_check_dt` (stdlib `timezone.utc` or pytz UTC zone).